

### Завдання 1: Виклик LLM з базовим промптом

**Мета:** навчитися викликати LLM через LangChain зі звичайним текстовим промптом.

**Що потрібно зробити:**

1. Створіть промпт, який дозволяє отримати інформацію простою мовою на тему "Квантові обчислення". Відповідь моделі повинна містити визначення, ключові переваги та поточні дослідження в цій галузі.

2. Обмежте відповідь до 200 символів і пропишіть в промпті, аби відповідь була короткою (це зекономить вам час і гроші на згенеровані токени).

3. Встановіть своє значення температури на власний розсуд (тут немає правильного чи неправильного значення) і напишіть коментарем, чому ви обрали саме таке значення для цього завдання.

**Вибір моделі:** можна скористатись як моделлю з HuggingFace, так і ChatGPT будь-якої версії, яка вам до вподоби і пасує за прайсингом. В обох випадках потрібно імпортувати відповідний клас з LangChain для виклику LLM за API.

**Мова запитів:** промпти можна писати як українською, так і англійською — орієнтуйтесь на те, де і для чого ви хочете потім використовувати цей проєкт. У розв'язках промпти — українською.

---

**🔐 Як безпечно зберігати і підвантажувати API-ключі**

API-токен потрібно зчитувати з безпечного джерела, а **не хардкодити в ноутбуці**. Якщо хтось отримає доступ до вашого ключа, він буде витрачати токени за ваш рахунок, а вам це не треба :)

Є кілька способів. Перший ми використовували на лекції, ще два для розширення вашого розуміння, як ще це можна зробити і що шлях не лише один. Для виконання цього ДЗ можете використовувати будь-який спосіб підвантаження ключів у ноутбук.

**Спосіб 1: Файл `creds.json` (рекомендований)**

Створіть файл `creds.json` з вашими ключами, завантажте його в Google Colab під час роботи, але **не здавайте** цей файл у ДЗ і **не комітьте** в git.

```python
import json
with open("creds.json") as f:
    creds = json.load(f)
api_key = creds["HF_TOKEN"]
```

**Спосіб 2: Google Colab Secrets**

У лівій панелі Colab натисніть іконку 🔑 (Secrets) → "Add new secret" → введіть назву (наприклад, `HF_TOKEN`) та значення ключа → увімкніть тогл доступу для ноутбука.

```python
from google.colab import userdata
api_key = userdata.get("HF_TOKEN")
```

Зручно тим, що ключ зберігається в акаунті і доступний у всіх ваших ноутбуках. Мінус — при кожній новій сесії потрібно перевірити, що доступ увімкнено.

**Спосіб 3: Google AI Studio (для Gemini API)**

Якщо працюєте з моделями Google Gemini, отримати безкоштовний API-ключ можна в [Google AI Studio](https://aistudio.google.com/app/apikey): увійдіть з Google-акаунтом → натисніть "Get API key" → "Create API key". Далі використовуйте ключ через будь-який із способів вище.



In [1]:
import json
with open("creds.json") as f:
    creds = json.load(f)
api_key = creds["TOKEN"]

### Завдання 2: Створення параметризованого промпта для генерації тексту
Тепер ми хочемо оновити попередній фукнціонал так, аби в промпт ми могли передавати тему як параметр. Для цього скористайтесь `PromptTemplate` з `langchain` і реалізуйте параметризований промпт та виклик моделі з ним.

Запустіть оновлений функціонал (промпт + модел) для пояснень про теми
- "Баєсівські методи в машинному навчанні"
- "Трансформери в машинному навчанні"
- "Explainable AI"

Виведіть результати відпрацювання моделі на екран.

In [2]:
! pip install -q langchain-huggingface

In [3]:
! pip freeze | findstr huggingface

huggingface_hub==1.28.0
langchain-huggingface==1.2.3


In [4]:
%pip install -U langchain_openai

Note: you may need to restart the kernel to use updated packages.


In [8]:
from langchain_core.prompts import PromptTemplate
from huggingface_hub import InferenceClient
from langchain_openai import ChatOpenAI

In [9]:
overal_temperature = 0.1

In [10]:
llm = ChatOpenAI(temperature=overal_temperature,
            model = 'gpt-4o-mini',
            api_key = api_key,
            max_tokens = 300)

In [11]:
llm

ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.7.6', 'langchain': '1.4.9', 'langchain-openai': '1.7.4'}}, profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x000002A5D39C8550>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000002A5D39F9090>, root_client=<openai.OpenAI object at 0x000002A5D395DC90>

In [32]:
prompt_1 = """
Поясни квантові обчислення простою мовою.
Вкажи визначення, ключові переваги
та сучасні напрямки досліджень.
Відповідь — не більше 200 символів.
"""

answer = llm.invoke(prompt_1).content

print(answer)

[{'type': 'text', 'text': 'Квантові обчислення — це обробка інформації за допомогою квантових бітів (кубітів). Вони швидші для складних задач, як криптографія та моделювання молекул. Досліджують стабільність кубітів і алгоритми.', 'annotations': [], 'id': 'msg_042f52a18580ccb2006af093c0139487d28a201833caf57b8f'}]


In [13]:
prompt = PromptTemplate(
    input_variables=["topic"],
    template="Лаконічно, але чітко поясни тему {topic}?",
)

In [14]:
print(prompt.format(topic="Баєсівські методи в машинному навчанні"))

Лаконічно, але чітко поясни тему Баєсівські методи в машинному навчанні?


In [15]:
print(llm.invoke(prompt.format(topic="Баєсівські методи в машинному навчанні")))

content='Баєсівські методи в машинному навчанні базуються на теорії ймовірностей, зокрема на теоремі Байєса. Вони використовують апостеріорні ймовірності для оцінки параметрів моделей і прийняття рішень на основі даних.\n\nОсновні принципи:\n\n1. **Теорема Байєса**: Вона дозволяє оновлювати ймовірність гіпотези (моделі) на основі нових даних. Формула виглядає так:\n   \\[\n   P(H|D) = \\frac{P(D|H) \\cdot P(H)}{P(D)}\n   \\]\n   де \\(P(H|D)\\) — апостеріорна ймовірність, \\(P(D|H)\\) — правдоподібність, \\(P(H)\\) — апріорна ймовірність, \\(P(D)\\) — нормалізуючий фактор.\n\n2. **Апріорні ймовірності**: Це початкові припущення про параметри моделі, які можуть бути отримані з попередніх знань або даних.\n\n3. **Правдоподібність**: Вимірює, наскільки добре модель пояснює спостережувані дані.\n\n4. **Регуляризація' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 300, 'prompt_tokens': 32, 'total_tokens': 332, 'completion_tokens_details': {'accep

In [16]:
print(llm.invoke(prompt.format(topic="Трансформери в машинному навчанні")))

content='Трансформери — це архітектура нейронних мереж, яка була вперше представлена в статті "Attention is All You Need" у 2017 році. Вона стала основою для багатьох сучасних моделей у обробці природної мови (NLP) та інших задачах.\n\nОсновні характеристики трансформерів:\n\n1. **Механізм уваги (Attention)**: Трансформери використовують механізм уваги, який дозволяє моделі зосереджуватися на різних частинах вхідних даних, оцінюючи їхню важливість. Це дозволяє ефективно обробляти контекст.\n\n2. **Паралелізм**: На відміну від рекурентних нейронних мереж (RNN), трансформери дозволяють обробляти всі елементи вхідної послідовності одночасно, що значно прискорює навчання.\n\n3. **Складна структура**: Трансформери складаються з енкодерів і декодерів. Енкодери перетворюють вхідні дані в контекстуальні представлення, а декодери генерують вихідні дані на основі цих представлень.\n\n4. **Універсальність**: Трансформери використовуються не лише в' additional_kwargs={'refusal': None} response_met

In [17]:
print(llm.invoke(prompt.format(topic="Explainable AI")))

content='Explainable AI (XAI) — це підрозділ штучного інтелекту, який зосереджується на розробці моделей і алгоритмів, здатних пояснювати свої рішення та дії. Основна мета XAI полягає в підвищенні прозорості, зрозумілості та довіри до систем штучного інтелекту, особливо в критичних сферах, таких як медицина, фінанси та право.\n\nКлючові аспекти XAI:\n\n1. **Прозорість**: Моделі повинні бути зрозумілими для користувачів, щоб вони могли зрозуміти, як і чому було прийнято певне рішення.\n2. **Довіра**: Пояснення допомагають користувачам довіряти системам AI, особливо коли результати можуть мати серйозні наслідки.\n3. **Відповідальність**: Зрозумілі рішення дозволяють легше виявляти помилки та упередження в алгоритмах.\n4. **Регуляція**: У деяких галузях існують вимоги до пояснювальності рішень, що робить XAI важливим для дотримання норм і стандартів.\n\nЗагалом, Explainable AI прагне зробити штучний інтел' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_



### Завдання 3: Використання агента для автоматизації процесів
Створіть агента, який допоможе автоматично шукати інформацію про останні наукові публікації в різних галузях. Наприклад, агент має знайти 5 останніх публікацій на тему штучного інтелекту.

**Кроки:**
1. Налаштуйте агента типу ReAct в LangChain для виконання автоматичних запитів.
2. Створіть промпт, який спрямовує агента шукати інформацію в інтернеті або в базах даних наукових публікацій.
3. Агент повинен видати список публікацій, кожна з яких містить назву, авторів і короткий опис.

Для взаємодії з пошуком там необхідно створити `Tool`. В лекції ми використовували `serpapi`. Можна продовжити користуватись ним, або обрати інше АРІ для пошуку (вони в тому числі є безкоштовні). Перелік різних АРІ, доступних в langchain, і орієнтир по вартості запитів можна знайти в окремому документі [тут](https://hannapylieva.notion.site/API-12994835849480a69b2adf2b8441cbb3?pvs=4).

Лишаю також нижче приклад використання одного з безкоштовних пошукових АРІ - DuckDuckGo (не потребує створення токена!)  - можливо він вам сподобається :)


In [19]:
%pip install arxiv

Note: you may need to restart the kernel to use updated packages.


In [20]:
from langchain_core.tools import tool
from langchain.agents import create_agent
import arxiv

In [21]:
@tool
def research_search(query: str) -> str:
    """Search arXiv for scientific papers and return metadata."""

    client = arxiv.Client()

    search = arxiv.Search(
        query=query,
        max_results=5,
        sort_by=arxiv.SortCriterion.SubmittedDate
    )

    papers = []

    for paper in client.results(search):
        papers.append({
            "title": paper.title,
            "authors": [a.name for a in paper.authors],
            "date": str(paper.published.date()),
            "summary": paper.summary[:5000],
            "url": paper.entry_id
        })

    return str(papers)

In [22]:
agent = create_agent(
    model=llm,
    tools=[research_search],
    system_prompt="""
    Ти науковий AI-асистент.

    Коли користувач просить знайти наукові публікації:

    1. ОБОВ'ЯЗКОВО виклич інструмент research_search.
    2. Використовуй пошуковий запит: cat:cs.AI
    3. Отримай 5 найновіших публікацій.
    4. Для кожної публікації вкажи:
       - Назву
       - Авторів
       - Дату
       - Короткий опис українською
       - Посилання

    Використовуй тільки дані з інструмента.
    Не вигадуй публікації.
    Якщо інструмент повернув результати,
    обов'язково представ їх користувачу.
    """
)

In [23]:
response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Знайди 5 найновіших наукових публікацій про штучний інтелект."
            }
        ]
    },
    config={"recursion_limit": 12}
)

print(response["messages"][-1].content)

Ось 5 найновіших наукових публікацій про штучний інтелект:

1. **SwarmEvolve: Latent Agent Swarms for Evolutionary Discovery and Optimization**
   - **Автори:** Qile Jiang, George Em Karniadakis
   - **Дата:** 2026-11-05
   - **Опис:** Публікація представляє SwarmEvolve, фреймворк, що використовує сотні агентів для еволюційного пошуку, що дозволяє швидше знаходити рішення математичних і обчислювальних задач, порівняно з традиційними методами.
   - **Посилання:** [arXiv:2611.08483v1](http://arxiv.org/abs/2611.08483v1)

2. **Scientist World: Grow a Research World for AI-Scientists**
   - **Автори:** Taiming Lu, Sachin Konan, Zhuang Liu
   - **Дата:** 2026-11-05
   - **Опис:** У цій публікації представлено "Світ науковців", де кожна наукова стаття стає виконуваним середовищем для AI-науковців, що дозволяє їм перевершувати людських дослідників у 74% випадків.
   - **Посил




### Завдання 4: Створення агента-помічника для вирішення бізнес-задач

Створіть агента, який допомагає вирішувати задачі бізнес-аналітики. Агент має допомогти користувачу створити прогноз по продажам на наступний рік враховуючи рівень інфляції і погодні умови. Агент має вміти використовувати Python і ходити в інтернет аби отримати актуальні дані.

**Кроки:**
1. Налаштуйте агента, який працюватиме з аналітичними даними, заданими текстом. Користувач пише

```
Ми експортуємо апельсини з Бразилії. В 2022 експортували 200т, в 2023 - 190т, в 2024 - 210т, в 2025 - 220т. Зроби оцінку скільки ми зможемо експортувати апельсинів в 2026 враховуючи погодні умови в Бразилії і попит на апельсини в світі виходячи з економічної ситуації.
```

2. Створіть запит до агента, що містить чітке завдання – видати результат бізнес аналізу або написати, що він не може цього зробити і запит користувача (просто може бути все одним повідомлленням).

3. Запустіть агента і проаналізуйте результати. Що можна покращити?


In [24]:
%pip install -U langchain-openai langchain numpy

Note: you may need to restart the kernel to use updated packages.


In [25]:
import numpy as np

In [26]:
with open("creds.json", "r", encoding="utf-8") as f:
    creds = json.load(f)

llm = ChatOpenAI(
    model="gpt-4.1-mini",
    api_key=api_key,
    temperature=0.1,
    max_tokens=2000,
    use_responses_api=True
)

llm_with_search = llm.bind_tools([
    {"type": "web_search_preview"}
])

In [27]:
@tool
def forecast_exports(
    demand_change_pct: float = 0.0,
    weather_change_pct: float = 0.0
) -> dict:
    """Calculate a 2026 export forecast with demand and weather adjustments."""

    years = np.array([2022, 2023, 2024, 2025])
    exports = np.array([200, 190, 210, 220])

    slope, intercept = np.polyfit(years - 2022, exports, 1)
    baseline = slope * 4 + intercept

    adjusted = baseline * (
        1 + demand_change_pct / 100
    ) * (
        1 + weather_change_pct / 100
    )

    return {
        "baseline_tonnes": round(float(baseline), 2),
        "adjusted_tonnes": round(float(adjusted), 2),
        "pessimistic_tonnes": round(float(adjusted * 0.9), 2),
        "optimistic_tonnes": round(float(adjusted * 1.1), 2)
    }

In [28]:
research = llm_with_search.invoke("""
Знайди актуальні дані за 2026 рік щодо:

1. Погодних умов у Бразилії та їхнього впливу
   на виробництво апельсинів.
2. Світового попиту на апельсини.
3. Інфляції та економічної ситуації.

Наведи джерела та дати.
Не вигадуй статистику.
Відповідай українською.
""")

print(research.content)

[{'id': 'ws_05cbfc37cc66e233006af092f6a00c87d297aacb197c4e7b7e', 'action': {'type': 'search', 'queries': ['погодні умови в Бразилії 2026 вплив на виробництво апельсинів'], 'query': 'погодні умови в Бразилії 2026 вплив на виробництво апельсинів'}, 'status': 'completed', 'type': 'web_search_call'}, {'type': 'text', 'text': 'На жаль, детальна інформація про погодні умови в Бразилії у 2026 році та їхній вплив на виробництво апельсинів наразі недоступна. Прогнози щодо світового попиту на апельсини також обмежені.\n\n**Економічна ситуація та інфляція:**\n\nУ 2026 році економіка США демонструє стійке зростання. Згідно з оцінками Бюро економічного аналізу США, реальний валовий внутрішній продукт (ВВП) зріс на 2,0% у першому кварталі 2026 року порівняно з попереднім кварталом. Основними чинниками цього зростання стали інвестиції, експорт, споживчі витрати та державні витрати. ([bea.gov](https://www.bea.gov/news/2026/gdp-advance-estimate-1st-quarter-2026?utm_source=openai))\n\nМіжнародний валютн

In [29]:
result = forecast_exports.invoke({
    "demand_change_pct": 0,
    "weather_change_pct": 0
})

print(result)

{'baseline_tonnes': 225.0, 'adjusted_tonnes': 225.0, 'pessimistic_tonnes': 202.5, 'optimistic_tonnes': 247.5}


In [30]:
report = llm.invoke(f"""
Ти Senior Business Analyst.

Історичні дані експорту апельсинів із Бразилії:
2022: 200 т
2023: 190 т
2024: 210 т
2025: 220 т

Результати Python-розрахунків:
{result}

Актуальна інформація з інтернету:
{research.content}

Підготуй бізнес-прогноз експорту на 2026 рік.

Обов'язково:
1. Проаналізуй історичну динаміку.
2. Оціни вплив погодних умов.
3. Оціни вплив світового попиту та інфляції.
4. Наведи базовий, оптимістичний і песимістичний сценарії.
5. Вкажи ризики й обмеження.
6. Якщо даних недостатньо, прямо скажи про це.

Не видавай умовні сценарії за статистично підтверджені.
Відповідай українською.
""")

print(report.content)

[{'type': 'text', 'text': '### Бізнес-прогноз експорту апельсинів із Бразилії на 2026 рік\n\n#### 1. Аналіз історичної динаміки\nЗа останні роки експорт апельсинів із Бразилії демонстрував помірне зростання:\n- 2022: 200 т\n- 2023: 190 т (зниження на 5%)\n- 2024: 210 т (зростання на 10,5%)\n- 2025: 220 т (зростання на 4,8%)\n\nТренд свідчить про загальне зростання обсягів експорту, незважаючи на незначні коливання. Середнє зростання за останні три роки становить близько 5-7% на рік.\n\n#### 2. Вплив погодних умов\nАктуальна інформація свідчить, що детальні дані про погодні умови в Бразилії на 2026 рік та їхній вплив на виробництво апельсинів наразі відсутні. Через це важко оцінити потенційні ризики, пов’язані з кліматичними факторами, які можуть суттєво вплинути на врожайність.\n\n#### 3. Вплив світового попиту та інфляції\n- Світовий попит на апельсини у 2026 році прогнозується обмежено через відсутність чітких даних.\n- Економічна ситуація у США, одному з ключових споживачів, характе